<!-- nav -->
[← 8 · Sublanguages](08_00_Sublanguages.ipynb) · [Home](../../README.md) · [8.14 · SystemVerilog Patterns →](08_14_SystemVerilog_Patterns.ipynb)

# 8.13 · SystemVerilog Types

Tutorial 8 met SystemVerilog's sublanguages once each: a cast's type, `int'(x)`, was a term of `SystemVerilog Types`
(08.1). This page takes that dialect on its own, from the data types a reader of IEEE 1800 knows (clauses 6 and 7) to
their models: a keyword type, a vector with packed dimensions, a structure, a union and an enumeration, each written in
the builders' fluent calls beside the text SystemVerilog writes, which the family's printer gives back. Then what is
done with them: validated, with one refused; read by the expressions that hold them, a cast, `$bits` and an assignment
pattern, whose evaluator converts a value to some of them; given no domain and no value of their own; and the editions
of Verilog and SystemVerilog that have each kind, read as data.

Like the rest of Part I, it starts from `dialects()`, a store with every dialect installed and defined. `name` and
`literal` write a name and a literal of SystemVerilog's `Expressions` (tutorials 3 and 4), wherever a type holds one: a
dimension's bounds are expressions. `refused` gives what an evaluator says of an expression, its value or the reason it
refuses it, as tutorial 8's does; `attributes` lists a form's attributes, `name=value`, a helper by its kind (tutorial
6, 06.5).

In [1]:
import assert from "node:assert/strict";
import * as SystemVerilog from "@mbse/expressions/SystemVerilog/Expressions";
import * as SVTypes from "@mbse/expressions/SystemVerilog/Types";
import { Domains, Standards, Symbolics, Terms } from "@mbse/expressions/Framework";
import { dialects } from "./toolkit.js";

const store = dialects();
const S = store.Expressions;
const Types = SVTypes.Expressions.DIALECT; // the dialect, as the framework reads it
const name = (spelling: string) => (d: any) => d.nameexpression((n: any) => n.name((a: any) => a.identifier((i: any) => i.spelling(spelling))));
const literal = (value: unknown) => (d: any) => d.literal((l: any) => l.value((v: any) => v.as_scalar(value)));

/** What `evaluate` says of `expression`: its value, or the reason it refuses it. */
function refused(evaluate: (expression: any) => unknown, expression: any): string {
  try {
    return String(evaluate(expression));
  } catch (error) {
    return (error as Error).message;
  }
}

/** A form's attributes, each `name=value`: a helper as its kind, helpers in brackets, a bool in lower case. */
function attributes(form: Terms.Form): string {
  const written = (value: unknown): string => Array.isArray(value) ? `[${value.map(written).join(", ")}]`
    : value instanceof Terms.Part ? value.kind : String(value);
  return [...form.attributes].map(([key, value]) => `${key}=${written(value)}`).join(", ");
}

<a id="T08.13.1"></a>
## 08.13.1 · A type keyword

```text
int        byte unsigned        real        string
```

The simplest data types are keywords: an integer atom (`byte`, `shortint`, `int`, `longint`, `integer`, `time`), signed
or unsigned; a non-integer type (`shortreal`, `real`, `realtime`); and the keyword types (`string`, `chandle`, `event`,
`void`). Each is a kind of `SystemVerilog Types`, `integeratomtype`, `nonintegertype` and `keywordtype`, its keyword and
its signing attributes, and each is the root of the dialect's holder, `S.SystemVerilog.Types.Expression`, which refines
the framework's `Expression` as every holder does (tutorial 3). The family's printer, `SystemVerilog.Text.ToText`, writes
any of its six dialects' terms (08.5 of tutorial 8):

In [2]:
const keywords: any[] = store.transaction(() => [
  (u: any) => u.integeratomtype((a: any) => a.keyword("int")),
  (u: any) => u.integeratomtype((a: any) => a.keyword("byte").signing("unsigned")),
  (u: any) => u.nonintegertype((n: any) => n.keyword("real")),
  (u: any) => u.keywordtype((k: any) => k.keyword("string")),
].map((root) => S.SystemVerilog.Types.Expression((e: any) => e.root(root))));
for (const held of keywords) {
  const form = Terms.form(Terms.term(held));
  console.log(SystemVerilog.Text.ToText(held).padEnd(14), form.kind.padEnd(15), attributes(form).padEnd(30), Terms.dialectOf(Terms.term(held)));
}
assert.deepEqual(keywords.map((held) => SystemVerilog.Text.ToText(held)), ["int", "byte unsigned", "real", "string"]);

int            integeratomtype keyword=int                    Expressions.SystemVerilog.Types.Term


byte unsigned  integeratomtype keyword=byte, signing=unsigned Expressions.SystemVerilog.Types.Term


real           nonintegertype  keyword=real                   Expressions.SystemVerilog.Types.Term


string         keywordtype     keyword=string                 Expressions.SystemVerilog.Types.Term


A keyword is one of a few the standard allows, and the dialect knows which: the choices are its kinds' own checks,
which validation reads (08.13.4).

<a id="T08.13.2"></a>
## 08.13.2 · Vectors and their dimensions

```text
logic signed [7:0]        bit [3:0]        logic [W-1:0]
```

A vector type, `bit`, `logic` or `reg`, is an `integervectortype`, its keyword and signing attributes again, and its
packed dimensions, each `[left:right]` a `rangedimension`. A dimension is no term: it is a helper, a value object the
type holds, which a form gives as a part (tutorial 6, 06.5). Its bounds are terms, and they are `Expressions`' terms,
expressions as any other: `W-1` is a binary expression whose left operand is a name. So a type can need a name from its
scope, as an expression does (tutorial 4):

In [3]:
const vectors: any[] = store.transaction(() => [
  (u: any) => u.integervectortype((v: any) => v.keyword("logic").signing("signed").dimensions([(d: any) => d.rangedimension((r: any) => r.left(literal(7n)).right(literal(0n)))])),
  (u: any) => u.integervectortype((v: any) => v.keyword("bit").dimensions([(d: any) => d.rangedimension((r: any) => r.left(literal(3n)).right(literal(0n)))])),
  (u: any) => u.integervectortype((v: any) => v.keyword("logic").dimensions([(d: any) => d.rangedimension((r: any) => r
    .left((b: any) => b.binaryexpression((o: any) => o.left(name("W")).operator("-").right(literal(1n)))).right(literal(0n)))])),
].map((root) => S.SystemVerilog.Types.Expression((e: any) => e.root(root))));
for (const held of vectors) {
  const form = Terms.form(Terms.term(held));
  console.log(SystemVerilog.Text.ToText(held).padEnd(19), attributes(form).padEnd(48), "needs",
    [...Symbolics.free(held, Types)].sort().join(", ") || "nothing");
}
const bounds = Terms.form(Terms.term(vectors[2])).arguments;
console.log("its bounds:", bounds.map((b: any) => `${SystemVerilog.Text.ToText(b)} (${Terms.kindOf(b)[0]} of ${Terms.dialectOf(b)})`).join(", "));
assert.deepEqual(vectors.map((held) => SystemVerilog.Text.ToText(held)), ["logic signed [7:0]", "bit [3:0]", "logic [W - 1:0]"]);

logic signed [7:0]  keyword=logic, signing=signed, dimensions=[rangedimension] needs nothing


bit [3:0]           keyword=bit, dimensions=[rangedimension]         needs nothing


logic [W - 1:0]     keyword=logic, dimensions=[rangedimension]       needs W


its bounds: W - 1 (binaryexpression of Expressions.SystemVerilog.Expressions.Term), 0 (literal of Expressions.SystemVerilog.Expressions.Term)


The printer writes `W - 1` with spaces, as it writes any binary expression. A range is one of five kinds of dimension,
read from the store as the branches of the dialect's union `Expressions.SystemVerilog.Types.Dimension`: `[7:0]`, a
size `[8]`, an unsized `[]`, an associative `[int]` or `[*]` and a queue `[$]`:

In [4]:
console.log(store.registered("Expressions.SystemVerilog.Types.Dimension").branches.map((branch: any) => branch.name).join(", "));

rangedimension, sizedimension, unsizeddimension, associativedimension, queuedimension


<a id="T08.13.3"></a>
## 08.13.3 · Structures, unions and enumerations: their members are data

```text
struct packed { logic [7:0] lo, hi; }
union tagged { void Invalid; int Valid; }
enum logic [1:0] {IDLE, RUN = 2}
word_t        type(x)
```

A structure or a union is a `structtype`, its keyword `struct` or `union`, `packed` or not, a tagged union's qualifier
`tagged`. Its members are data, not terms (stage 6, "Q7"): each a `structmember` helper holding its type, a term of this
dialect again, and its declarators, each a `variabledeclarator` naming a member. An enumeration is an `enumtype`, its
base type a term, its members `enummember` helpers, each a name and perhaps a value, an expression. A type named by a
typedef is a `namedtype`, its name `Expressions`' name helper (tutorial 4), and `type(x)`, a type reference, holds the
expression whose type it is:

In [5]:
const byte_ = (t: any) => t.integervectortype((v: any) => v.keyword("logic").dimensions([(d: any) => d.rangedimension((r: any) => r.left(literal(7n)).right(literal(0n)))]));
const declared = (spelling: string) => (d: any) => d.name((i: any) => i.spelling(spelling)); // a declarator, of its name
const aggregates: any[] = store.transaction(() => [
  (u: any) => u.structtype((s: any) => s.keyword("struct").packed(true).members([
    (m: any) => m.type(byte_).declarators([declared("lo"), declared("hi")])])),
  (u: any) => u.structtype((s: any) => s.keyword("union").qualifier("tagged").members([
    (m: any) => m.type((t: any) => t.keywordtype((k: any) => k.keyword("void"))).declarators([declared("Invalid")]),
    (m: any) => m.type((t: any) => t.integeratomtype((a: any) => a.keyword("int"))).declarators([declared("Valid")])])),
  (u: any) => u.enumtype((n: any) => n
    .base((t: any) => t.integervectortype((v: any) => v.keyword("logic").dimensions([(d: any) => d.rangedimension((r: any) => r.left(literal(1n)).right(literal(0n)))])))
    .members([(m: any) => m.name((i: any) => i.spelling("IDLE")), (m: any) => m.name((i: any) => i.spelling("RUN")).value(literal(2n))])),
  (u: any) => u.namedtype((t: any) => t.name((n: any) => n.identifier((i: any) => i.spelling("word_t")))),
  (u: any) => u.typereference((t: any) => t.operand(name("x"))),
].map((root) => S.SystemVerilog.Types.Expression((e: any) => e.root(root))));
for (const held of aggregates) {
  const form = Terms.form(Terms.term(held));
  console.log(SystemVerilog.Text.ToText(held).padEnd(42), form.kind);
  console.log(" ", attributes(form) || "-", "| its terms:",
    form.arguments.filter((a: any) => a != null).map((a: any) => Terms.kindOf(a)[0]).join(", ") || "none");
}
assert.deepEqual(aggregates.map((held) => SystemVerilog.Text.ToText(held)), [
  "struct packed { logic [7:0] lo, hi; }", "union tagged { void Invalid; int Valid; }",
  "enum logic [1:0] {IDLE, RUN = 2}", "word_t", "type(x)"]);

struct packed { logic [7:0] lo, hi; }      structtype


  keyword=struct, packed=true, members=[structmember] | its terms: integervectortype


union tagged { void Invalid; int Valid; }  structtype


  keyword=union, qualifier=tagged, members=[structmember, structmember] | its terms: keywordtype, integeratomtype


enum logic [1:0] {IDLE, RUN = 2}           enumtype


  members=[enummember, enummember] | its terms: integervectortype, literal


word_t                                     namedtype


  name=identifier | its terms: none


type(x)                                    typereference


  - | its terms: nameexpression


A member's type is a subterm the structure's form gives among its arguments (`integervectortype`), while the member
itself, and its declarators' names, are in its attributes, as parts. A member's type may be any type, a structure
within a structure among them, so the dialect's `Term` holds them all.

<a id="T08.13.4"></a>
## 08.13.4 · Validated, and one refused

`validate` checks a type as it checks any term (tutorial 4): what each kind requires, each keyword among its choices,
each name's spelling, and every name its expressions read, which the caller must bind (the second column).
`logic [W-1:0]` needs `W`; given it, it is valid. A structure with a qualifier is refused, since only a union is
`tagged` or `soft` (IEEE 1800 7.3); the union above, the same members under `union`, passes:

In [6]:
const tagged_struct = store.transaction(() => S.SystemVerilog.Types.Expression((e: any) => e.root((u: any) => u.structtype((s: any) => s
  .keyword("struct").qualifier("tagged").members([
    (m: any) => m.type((t: any) => t.integeratomtype((a: any) => a.keyword("int"))).declarators([declared("Valid")])])))));
for (const [held, bound] of [[vectors[2], []], [vectors[2], ["W"]], [tagged_struct, []], [aggregates[1], []]] as [any, string[]][]) {
  console.log(SystemVerilog.Text.ToText(held).padEnd(42), (bound.join(", ") || "-").padEnd(6), Types.validate(held, bound).join("; ") || "valid");
}
assert.deepEqual(Types.validate(tagged_struct), ["a StructType with a qualifier is a union"]);

logic [W - 1:0]                            -      dimensions 0 left: left: nameexpression 'W' is not bound


logic [W - 1:0]                            W      valid


struct tagged { int Valid; }               -      a StructType with a qualifier is a union


union tagged { void Invalid; int Valid; }  -      valid


Validation reports; it does not repair: the refused structure is kept, and printed as written, as tutorial 4 kept a name
its language could not spell.

<a id="T08.13.5"></a>
## 08.13.5 · Read by the expressions that hold it

```text
int'(x)    byte'(x)    real'(x)    logic [7:0]'(x)    $bits(logic [7:0])    pair_t'{1, 2}
```

A type stands where SystemVerilog's expressions put one: a cast's type, a system call's argument (`$bits` of a type), an
assignment pattern's type. Which properties of `SystemVerilog`'s kinds hold this dialect's `Term` is read from the store,
as tutorial 8 did (08.3). Its host, `SystemVerilog`'s `Expressions`, reads the type through the dialects it reads
(tutorial 8, 08.1), and validates the whole tree:

In [7]:
const union = store.registered(Types.union);
const host = store.registered(SystemVerilog.Expressions.DIALECT.union).branches as any[];
console.log(host.flatMap((b) => [...b.schema.properties.entries()]
  .filter(([, prop]: [string, any]) => prop.schema === union || prop.schema.element === union)
  .map(([p]: [string, any]) => `${b.name}.${p}`)).join(", "));
const held: any[] = store.transaction(() => [
  (u: any) => u.castexpression((c: any) => c.type((t: any) => t.integeratomtype((a: any) => a.keyword("int"))).value(name("x"))),
  (u: any) => u.castexpression((c: any) => c.type((t: any) => t.integeratomtype((a: any) => a.keyword("byte"))).value(name("x"))),
  (u: any) => u.castexpression((c: any) => c.type((t: any) => t.nonintegertype((n: any) => n.keyword("real"))).value(name("x"))),
  (u: any) => u.castexpression((c: any) => c.type(byte_).value(name("x"))),
  (u: any) => u.systemcall((c: any) => c.name("$bits").arguments([byte_])),
  (u: any) => u.assignmentpattern((a: any) => a.type((t: any) => t.namedtype((n: any) => n.name(
    (m: any) => m.identifier((i: any) => i.spelling("pair_t"))))).items([literal(1n), literal(2n)])),
].map((root) => S.SystemVerilog.Expressions.Expression((e: any) => e.root(root))));
for (const expression of held) {
  const form = Terms.form(Terms.term(expression));
  const types = form.arguments.filter((a: any) => a != null && Terms.dialectOf(a) === Types.union).map((a: any) => Terms.kindOf(a)[0]);
  console.log(SystemVerilog.Text.ToText(expression).padEnd(19), form.kind.padEnd(18), "holds", types.join(", "),
    "| problems:", SystemVerilog.Expressions.DIALECT.validate(expression, ["x"]).join(", ") || "none");
}
assert.deepEqual(held.map((e) => SystemVerilog.Text.ToText(e)), [
  "int'(x)", "byte'(x)", "real'(x)", "logic [7:0]'(x)", "$bits(logic [7:0])", "pair_t'{1, 2}"]);

assignmentpattern.type, patternitem.key, systemcall.arguments, castexpression.type, streamingconcatenation.slice


int'(x)             castexpression     holds integeratomtype | problems: none


byte'(x)            castexpression     holds integeratomtype | problems: none


real'(x)            castexpression     holds nonintegertype | problems: none


logic [7:0]'(x)     castexpression     holds integervectortype | problems: none


$bits(logic [7:0])  systemcall         holds integervectortype | problems: none


pair_t'{1, 2}       assignmentpattern  holds namedtype | problems: none


Its evaluator converts a value to a type it has a stored domain for, by name: SystemVerilog's own domains (tutorial 2,
02.4), `int` and `byte` here, so `int'(300)` is 300 and `byte'(300)` is 44, 300 wrapped to 8 bits. Its inference reads
the same domain, `real`'s too. A type it has no stored domain for, a vector with dimensions, a named type, is refused by
the name of its kind and the dialect that owns it, as tutorial 8 showed every sublanguage's kinds are; its inference
gives `any`, no domain known, until each term's domain is inferred as data
([CAP-232: domains inferred for terms](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-232.md)):

In [8]:
const [dialect, evaluate] = [SystemVerilog.Expressions.DIALECT, SystemVerilog.Evaluators.OfAny];
const int_: any = Domains.domains(store.Instances.Expressions.SystemVerilog.Expressions).get("int");
for (const expression of held.slice(0, 4)) {
  console.log(SystemVerilog.Text.ToText(expression).padEnd(19), dialect.infer(expression, { x: int_ }).name().padEnd(5),
    expression !== held[2] ? refused((e) => evaluate(e, { x: 300n }), expression) : "-");
}
console.log(SystemVerilog.Text.ToText(held[4]).padEnd(19), "".padEnd(5), refused((e) => evaluate(e), held[4]));

int'(x)             int   32'sb00000000000000000000000100101100


byte'(x)            byte  8'sb00101100


real'(x)            real  -


logic [7:0]'(x)     any   SystemVerilog Types integervectortype is not evaluated


$bits(logic [7:0])        SystemVerilog Types integervectortype is not evaluated


(`real'(300)` evaluates too, to the float 300.0; it is left out here, since each language writes a float its own way.)

<a id="T08.13.6"></a>
## 08.13.6 · A type has no domain and no value of its own

A type names a set of values; it is not one. So a term of `SystemVerilog Types` has no domain to infer, `any`, and
SystemVerilog's evaluator refuses it by its kind's name, held alone or wherever it stands. What it names is stored
data: the domain `int`, read by name, is the one the cast's inference gave above, 32 bits, signed, two-state:

In [9]:
console.log(Types.infer(keywords[0]).name(), "|", refused((e) => evaluate(e), keywords[0]), "|", refused((e) => evaluate(e), aggregates[0]));
const properties = Domains.properties(int_);
console.log("int:", Domains.kindOf(int_).slice(0, 2).join("."), String(properties.get("width")), "bits,", properties.get("signed") ? "signed" : "unsigned");
assert.ok(Types.infer(keywords[0]) === Domains.Anything && (dialect.infer(held[0], { x: int_ }) as any).domain === int_);

any | SystemVerilog Types integeratomtype is not evaluated | SystemVerilog Types structtype is not evaluated


int: systemverilog.bit 32 bits, signed


<a id="T08.13.7"></a>
## 08.13.7 · Its editions are data

The dialect covers Verilog's and SystemVerilog's editions, IEEE Std 1364-1995 to 1364-2005 and IEEE Std 1800-2005 to
1800-2023, each found by its designation or its common name, and each records the kinds it has
([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md)):
Verilog's have the keyword and vector types and the implicit type (`signed'(x)`), and IEEE 1800-2005 added named types,
virtual interfaces, structures, unions, enumerations and type references:

In [10]:
const singleton = store.Instances.Expressions.SystemVerilog.Types;
const kinds = new Map((store.registered(Types.union).branches as any[]).map((branch) => [branch.schema, branch.name]));
for (const [designation, edition] of Standards.standards(singleton)) {
  console.log(designation.padEnd(19), Standards.tags(edition)[1]!.padEnd(19), Standards.kinds(edition).length, "kinds");
}
const verilog = Standards.kinds(Standards.find(singleton, "Verilog-2005")!);
const added = Standards.kinds(Standards.find(singleton, "SystemVerilog-2005")!).filter((k) => !verilog.includes(k)).map((k) => kinds.get(k));
console.log("added in IEEE 1800-2005:", added.join(", "));
assert.ok(Standards.find(singleton, "IEEE Std 1800-2023") === Standards.find(singleton, "SystemVerilog-2023"));

IEEE Std 1364-1995  Verilog-1995        5 kinds


IEEE Std 1364-2001  Verilog-2001        5 kinds


IEEE Std 1364-2005  Verilog-2005        5 kinds


IEEE Std 1800-2005  SystemVerilog-2005  10 kinds


IEEE Std 1800-2009  SystemVerilog-2009  10 kinds


IEEE Std 1800-2012  SystemVerilog-2012  10 kinds


IEEE Std 1800-2017  SystemVerilog-2017  10 kinds


IEEE Std 1800-2023  SystemVerilog-2023  10 kinds


added in IEEE 1800-2005: namedtype, virtualinterfacetype, structtype, enumtype, typereference


An edition records the kinds it has, not each keyword: `integeratomtype` is in Verilog-1995's, though `int` and `byte`
arrived with SystemVerilog.

## What you learned

- **A data type is a term of `SystemVerilog Types`**: a keyword, a vector and its dimensions, a structure, a union, an
  enumeration, a named type or a type reference, each held alone by the dialect's holder and printed by the family's
  printer.
- **Its parts are data where they are no type**: dimensions, members and declarators are helpers; a dimension's bounds
  and a member's value are `Expressions`' terms, so a type can need a name from its scope.
- **It is validated as any term**, its keywords' choices and its kinds' rules among its checks, and refused by name,
  never repaired.
- **Its host reads it**: a cast, `$bits` and an assignment pattern hold it; the evaluator converts to a type it has a
  stored domain for, and refuses the others by name.
- **A type has no domain or value of its own**, and the editions that have each of its kinds are data.

Next, the patterns a `matches` compares a value with: [SystemVerilog Patterns](08_14_SystemVerilog_Patterns.ipynb).

<!-- nav -->
[← 8 · Sublanguages](08_00_Sublanguages.ipynb) · [Home](../../README.md) · [8.14 · SystemVerilog Patterns →](08_14_SystemVerilog_Patterns.ipynb)